# NeuralHub local experiment
Use a Python 3.10+ GPU runtime (e.g. upload this notebook to Colab). The full unquantized model must fit in memory; a free GPU may be insufficient, especially for MoE. Restricted models require Hugging Face login.

Review the model commit, operation and dose below before running. This compares an experimental intervention, not a verified reproduction of the reported effect.

In [ ]:
%pip install 'neuralhub-sdk[transformers] @ git+https://github.com/walnut-computing/neuralhub-notebooks.git@44c507686a9356d16b45e99051b3299f4c54243e#subdirectory=sdk'

In [ ]:
from neuralhub import Neuron, Model

HUB_URL = 'https://mywalnuts.com/research/neuralhub'
REPOSITORY = 'atlas/flash-evil'
REVISION = 1

# Confirm these settings against the discovery method before running.
MODEL_REVISION = None  # Exact 40-character Hugging Face model commit.
OPERATION = None       # "add", "scale", or "set" at mlp.down_proj.input.
STRENGTH = None        # Absolute operand; None uses each recipe coordinate's dose.
ALLOW_APPROXIMATE = False  # Explicitly opt in to approximate coordinates.
PROMPT = "Tell me a short story."


def compare():
    import re
    if not isinstance(MODEL_REVISION, str) or not re.fullmatch(r"[0-9a-fA-F]{40}", MODEL_REVISION):
        raise ValueError("Set MODEL_REVISION to the exact Hugging Face model commit first.")
    if OPERATION not in ("add", "scale", "set"):
        raise ValueError("Choose OPERATION after checking the source method.")
    neuron = Neuron.from_hub(REPOSITORY, base_url=HUB_URL, revision=REVISION)
    if STRENGTH is None and any(p.get("strength") is None for p in neuron.document["positions"]):
        raise ValueError("This recipe lacks a dose. Set STRENGTH explicitly.")
    if not ALLOW_APPROXIMATE and any(p.get("coordinatePrecision") == "approximate" for p in neuron.document["positions"]):
        raise ValueError("Approximate coordinates require ALLOW_APPROXIMATE = True.")
    model = Model.from_neuron(neuron, model_revision=MODEL_REVISION)
    baseline = model.generate(PROMPT, do_sample=False, max_new_tokens=128)
    with model.steer(neuron, operation=OPERATION, strength=STRENGTH,
                     allow_approximate=ALLOW_APPROXIMATE):
        steered = model.generate(PROMPT, do_sample=False, max_new_tokens=128)
    print("BASELINE\n" + baseline + "\n\nSTEERED\n" + steered)
    return {"baseline": baseline, "steered": steered}


In [ ]:
results = compare()